In [25]:
# NML TEMP & HUM FULL IMPORTER

import os
import re
import warnings
import pandas as pd
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")

try:
    from docx import Document
except Exception as e:
    raise ImportError(
        "Missing dependency: python-docx.\n"
        "Install with:\n"
        "  pip install python-docx\n"
        f"\nOriginal error: {e}"
    )


# DB CONFIG

DB_CONFIG = {
    "drivername": "postgresql+psycopg2",
    "username": "postgres",
    "password": "Volkswagengolf2025",
    "host": "localhost",
    "port": 5432,
    "database": "NML_db",
}
DB_URL = (
    f"{DB_CONFIG['drivername']}://{DB_CONFIG['username']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

IMPORT_FOLDER = r"S:\CERTS 2026\Temp & Hum"


# HELPERS

def _norm(s):
    return re.sub(r"\s+", " ", str(s or "")).strip().lower()

def clean_text_value(x):
    if x is None:
        return None
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    return s if s else None

def safe_float(x):
    if x is None:
        return None
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return None
    s = s.replace(",", "")
    s = s.replace("±", "")
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
    if not m:
        return None
    try:
        return float(m.group(0))
    except Exception:
        return None

def safe_sql_date_from_text(x):
    x = clean_text_value(x)
    if not x:
        return None
    try:
        dt = pd.to_datetime(x, dayfirst=True, errors="coerce")
        if pd.notna(dt):
            return dt.strftime("%Y-%m-%d")
    except Exception:
        pass
    return None

def looks_like_label(text):
    if not text:
        return False
    t = _norm(text)
    known_labels = {
        "certificate number",
        "certificate no",
        "cert no",
        "instrument",
        "instrument type",
        "manufacturer",
        "make",
        "model",
        "serial number",
        "instrument serial number",
        "serial no",
        "id number",
        "id no",
        "id",
        "date received",
        "date of calibration",
        "calibration date",
        "nml procedure",
        "procedure number",
        "calibration procedure number",
        "comments",
    }
    return t in known_labels

def is_probably_bad_value(val):
    if not val:
        return True
    v = _norm(val)
    bad_exact = {
        "model",
        "serial number",
        "instrument serial number",
        "id number",
        "id",
        "id no",
        "manufacturer",
        "instrument",
        "date received",
        "date of calibration",
        "calibration date",
    }
    return v in bad_exact


# DOCX WALKERS

def iter_tables(parent):
    if hasattr(parent, "tables"):
        for t in parent.tables:
            yield t
            for r in t.rows:
                for c in r.cells:
                    yield from iter_tables(c)

def table_rows(t):
    rows = []
    for r in t.rows:
        rows.append([c.text.strip().replace("\n", " ") for c in r.cells])
    return rows


# FILE FILTER

def is_valid_cert_file(filename):
    name = filename.lower().strip()

    if name.startswith("~$"):
        return False

    blocked_words = [
        "draft",
        "copy",
        "template",
        "test",
        "old",
        "backup",
        "notes",
        "sample",
    ]
    if any(word in name for word in blocked_words):
        return False

    return True


# FRONT PAGE / LABEL EXTRACTION

def extract_doc_lines(doc):
    lines = []

    for p in doc.paragraphs:
        txt = clean_text_value(p.text)
        if txt:
            lines.append(txt)

    for t in iter_tables(doc):
        rows = table_rows(t)
        for r in rows:
            for c in r:
                txt = clean_text_value(c)
                if txt:
                    lines.append(txt)

    return lines

def extract_from_same_line(text, label_variants):
    for lbl in label_variants:
        pattern = rf"^\s*{re.escape(lbl)}\s*[:\-]?\s*(.+?)\s*$"
        m = re.match(pattern, text, flags=re.IGNORECASE)
        if m:
            val = clean_text_value(m.group(1))
            if val and not is_probably_bad_value(val):
                return val
    return None

def extract_label_value(lines, label_variants, max_lookahead=3):
    label_set = {_norm(x) for x in label_variants}

    for line in lines:
        val = extract_from_same_line(line, label_variants)
        if val:
            return val

    for i, line in enumerate(lines):
        if _norm(line) in label_set:
            for j in range(i + 1, min(i + 1 + max_lookahead, len(lines))):
                candidate = clean_text_value(lines[j])
                if not candidate:
                    continue
                if looks_like_label(candidate):
                    continue
                if is_probably_bad_value(candidate):
                    continue
                return candidate

    return None

def extract_certificate_number(doc, file_name):
    lines = extract_doc_lines(doc)

    cert = extract_label_value(
        lines,
        ["Certificate Number", "Certificate No", "Cert No"],
        max_lookahead=3
    )
    if cert:
        return cert

    return os.path.splitext(file_name)[0]

def extract_docx_metadata_fields(doc, file_name):
    lines = extract_doc_lines(doc)

    certificate_number = extract_certificate_number(doc, file_name)

    instrument_type = extract_label_value(lines, ["Instrument", "Instrument Type"])
    manufacturer = extract_label_value(lines, ["Manufacturer", "Make"])
    model = extract_label_value(lines, ["Model"])
    serial_number = extract_label_value(
        lines,
        ["Serial Number", "Instrument Serial Number", "Serial No"]
    )
    id_number = extract_label_value(lines, ["ID Number", "ID No", "ID"])
    date_received = extract_label_value(lines, ["Date Received"])
    calibration_date = extract_label_value(lines, ["Date of Calibration", "Calibration Date"])
    calibration_procedure_number = extract_label_value(
        lines,
        ["NML Procedure", "Procedure Number", "Calibration Procedure Number"]
    )

    return {
        "certificate_number": clean_text_value(certificate_number),
        "instrument_type": clean_text_value(instrument_type),
        "manufacturer": clean_text_value(manufacturer),
        "model": clean_text_value(model),
        "serial_number": clean_text_value(serial_number),
        "id_number": clean_text_value(id_number),
        "date_received": safe_sql_date_from_text(date_received),
        "calibration_date": safe_sql_date_from_text(calibration_date),
        "calibration_procedure_number": clean_text_value(calibration_procedure_number),
        "comments": None,
        "lab_type": "Temp & Hum",
        "file_name": file_name,
    }


# METADATA TABLE SETUP

def ensure_metadata_columns(engine):
    with engine.begin() as conn:
        conn.execute(text("""
            ALTER TABLE calibration_metadata
            ADD COLUMN IF NOT EXISTS model TEXT;
        """))
        conn.execute(text("""
            ALTER TABLE calibration_metadata
            ADD COLUMN IF NOT EXISTS serial_number TEXT;
        """))
        conn.execute(text("""
            ALTER TABLE calibration_metadata
            ADD COLUMN IF NOT EXISTS id_number TEXT;
        """))

def get_existing_metadata_row(engine, file_name=None, certificate_number=None):
    file_stem = os.path.splitext(file_name)[0] if file_name else None

    with engine.connect() as conn:
        if file_name:
            row = conn.execute(text("""
                SELECT id, certificate_number, file_name
                FROM calibration_metadata
                WHERE file_name = :file_name
                LIMIT 1
            """), {"file_name": file_name}).fetchone()
            if row:
                return row

        if certificate_number:
            row = conn.execute(text("""
                SELECT id, certificate_number, file_name
                FROM calibration_metadata
                WHERE certificate_number = :certificate_number
                LIMIT 1
            """), {"certificate_number": certificate_number}).fetchone()
            if row:
                return row

        if file_stem:
            row = conn.execute(text("""
                SELECT id, certificate_number, file_name
                FROM calibration_metadata
                WHERE certificate_number = :file_stem
                LIMIT 1
            """), {"file_stem": file_stem}).fetchone()
            if row:
                return row

    return None

def upsert_metadata_from_docx(engine, file_path):
    file_name = os.path.basename(file_path)
    doc = Document(file_path)
    meta = extract_docx_metadata_fields(doc, file_name)

    existing = get_existing_metadata_row(
        engine,
        file_name=file_name,
        certificate_number=meta["certificate_number"]
    )

    with engine.begin() as conn:
        if existing:
            metadata_id = int(existing[0])

            conn.execute(text("""
                UPDATE calibration_metadata
                SET
                    certificate_number = COALESCE(:certificate_number, certificate_number),
                    file_name = COALESCE(:file_name, file_name),
                    lab_type = COALESCE(:lab_type, lab_type),
                    manufacturer = COALESCE(:manufacturer, manufacturer),
                    instrument_type = COALESCE(:instrument_type, instrument_type),
                    serial_number = COALESCE(:serial_number, serial_number),
                    model = COALESCE(:model, model),
                    id_number = COALESCE(:id_number, id_number),
                    calibration_date = COALESCE(CAST(:calibration_date AS DATE), calibration_date),
                    date_received = COALESCE(CAST(:date_received AS DATE), date_received),
                    calibration_procedure_number = COALESCE(:calibration_procedure_number, calibration_procedure_number),
                    comments = COALESCE(:comments, comments)
                WHERE id = :metadata_id
            """), {
                **meta,
                "metadata_id": metadata_id,
            })

            return metadata_id, meta["certificate_number"], doc, meta, "updated"

        else:
            row = conn.execute(text("""
                INSERT INTO calibration_metadata (
                    certificate_number,
                    file_name,
                    lab_type,
                    manufacturer,
                    instrument_type,
                    serial_number,
                    model,
                    id_number,
                    calibration_date,
                    date_received,
                    calibration_procedure_number,
                    comments
                )
                VALUES (
                    :certificate_number,
                    :file_name,
                    :lab_type,
                    :manufacturer,
                    :instrument_type,
                    :serial_number,
                    :model,
                    :id_number,
                    CAST(:calibration_date AS DATE),
                    CAST(:date_received AS DATE),
                    :calibration_procedure_number,
                    :comments
                )
                RETURNING id
            """), meta).fetchone()

            metadata_id = int(row[0])
            return metadata_id, meta["certificate_number"], doc, meta, "created"


# RESULT TABLE DETECTION + PARSING

def classify_table(header_cells):
    h = " ".join([_norm(x) for x in header_cells if x is not None])

    if ("reference relative humidity" in h or "reference rh" in h) and ("unit under test relative humidity" in h or "uut relative humidity" in h):
        return "HUM_CHAMBER_TEMP"

    if ("reference temperature" in h) and ("unit under test temperature" in h or "uut temperature" in h):
        return "TEMP_CHAMBER_RH"

    if ("reference reading" in h) and ("uut reading" in h or "unit under test reading" in h):
        return "TEMP_SIMPLE"

    return None

def parse_results_from_docx(doc, identity):
    records = []
    all_tables = list(iter_tables(doc))

    for t in all_tables:
        rows = table_rows(t)
        if not rows or len(rows) < 2:
            continue

        header = rows[0]
        ttype = classify_table(header)
        if not ttype:
            continue

        header_norm = [_norm(x) for x in header]

        def find_col_contains(*subs):
            for i, hn in enumerate(header_norm):
                if all(s in hn for s in subs):
                    return i
            return None

        if ttype == "HUM_CHAMBER_TEMP":
            col_ch_temp = find_col_contains("chamber", "temperature")
            col_ref = find_col_contains("reference", "humidity")
            col_uut = find_col_contains("unit under test", "humidity")
            if col_uut is None:
                col_uut = find_col_contains("uut", "humidity")
            col_corr = find_col_contains("correction")
            col_unc = find_col_contains("uncertainty")

            if col_ref is None or col_uut is None:
                continue

            for r in rows[1:]:
                ref = safe_float(r[col_ref]) if col_ref < len(r) else None
                uut = safe_float(r[col_uut]) if col_uut < len(r) else None
                if ref is None and uut is None:
                    continue

                chamber_temp = safe_float(r[col_ch_temp]) if (col_ch_temp is not None and col_ch_temp < len(r)) else None
                corr = safe_float(r[col_corr]) if (col_corr is not None and col_corr < len(r)) else (ref - uut if ref is not None and uut is not None else None)
                unc = safe_float(r[col_unc]) if (col_unc is not None and col_unc < len(r)) else None

                records.append({
                    "meas_type": "Humidity",
                    "chamber_temperature_c": chamber_temp,
                    "chamber_rh": None,
                    "reference_reading": ref,
                    "uut_reading": uut,
                    "correction": corr,
                    "uncertainty": unc,
                    "unit": "%RH",
                    "serial_number": identity.get("serial_number"),
                    "model": identity.get("model"),
                    "id_number": identity.get("id_number"),
                })

        elif ttype == "TEMP_CHAMBER_RH":
            col_ch_rh = find_col_contains("chamber", "humidity")
            col_ref = find_col_contains("reference", "temperature")
            col_uut = find_col_contains("unit under test", "temperature")
            if col_uut is None:
                col_uut = find_col_contains("uut", "temperature")
            col_corr = find_col_contains("correction")
            col_unc = find_col_contains("uncertainty")

            if col_ref is None or col_uut is None:
                continue

            for r in rows[1:]:
                ref = safe_float(r[col_ref]) if col_ref < len(r) else None
                uut = safe_float(r[col_uut]) if col_uut < len(r) else None
                if ref is None and uut is None:
                    continue

                chamber_rh = safe_float(r[col_ch_rh]) if (col_ch_rh is not None and col_ch_rh < len(r)) else None
                corr = safe_float(r[col_corr]) if (col_corr is not None and col_corr < len(r)) else (ref - uut if ref is not None and uut is not None else None)
                unc = safe_float(r[col_unc]) if (col_unc is not None and col_unc < len(r)) else None

                records.append({
                    "meas_type": "Temperature",
                    "chamber_temperature_c": None,
                    "chamber_rh": chamber_rh,
                    "reference_reading": ref,
                    "uut_reading": uut,
                    "correction": corr,
                    "uncertainty": unc,
                    "unit": "°C",
                    "serial_number": identity.get("serial_number"),
                    "model": identity.get("model"),
                    "id_number": identity.get("id_number"),
                })

        elif ttype == "TEMP_SIMPLE":
            flat_2rows = " ".join([" ".join(rows[0]), " ".join(rows[1])]).lower()
            if ("°c" not in flat_2rows) and ("(c)" not in flat_2rows) and ("degc" not in flat_2rows):
                continue

            col_set = find_col_contains("set")
            if col_set is None:
                col_set = find_col_contains("temperature")
            if col_set is None:
                col_set = 0

            col_ref = find_col_contains("reference", "reading")
            col_uut = find_col_contains("uut", "reading")
            if col_uut is None:
                col_uut = find_col_contains("unit under test", "reading")
            col_corr = find_col_contains("correction")
            col_unc = find_col_contains("uncertainty")

            if col_ref is None or col_uut is None:
                continue

            for r in rows[1:]:
                ref = safe_float(r[col_ref]) if col_ref < len(r) else None
                uut = safe_float(r[col_uut]) if col_uut < len(r) else None
                if ref is None and uut is None:
                    continue

                setpoint = safe_float(r[col_set]) if (col_set is not None and col_set < len(r)) else None
                corr = safe_float(r[col_corr]) if (col_corr is not None and col_corr < len(r)) else (ref - uut if ref is not None and uut is not None else None)
                unc = safe_float(r[col_unc]) if (col_unc is not None and col_unc < len(r)) else None

                records.append({
                    "meas_type": "Temperature",
                    "chamber_temperature_c": setpoint,
                    "chamber_rh": None,
                    "reference_reading": ref,
                    "uut_reading": uut,
                    "correction": corr,
                    "uncertainty": unc,
                    "unit": "°C",
                    "serial_number": identity.get("serial_number"),
                    "model": identity.get("model"),
                    "id_number": identity.get("id_number"),
                })

    return records


# RESULTS TABLE SETUP

def ensure_results_schema(engine):
    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS temp_hum_results (
                id SERIAL PRIMARY KEY,
                metadata_id INTEGER REFERENCES calibration_metadata(id) ON DELETE CASCADE,
                certificate_number TEXT,
                meas_type TEXT,
                chamber_temperature_c DOUBLE PRECISION,
                chamber_rh DOUBLE PRECISION,
                reference_reading DOUBLE PRECISION,
                uut_reading DOUBLE PRECISION,
                correction DOUBLE PRECISION,
                uncertainty DOUBLE PRECISION,
                unit TEXT,
                serial_number TEXT,
                model TEXT,
                id_number TEXT
            );
        """))

        conn.execute(text("ALTER TABLE temp_hum_results ADD COLUMN IF NOT EXISTS serial_number TEXT;"))
        conn.execute(text("ALTER TABLE temp_hum_results ADD COLUMN IF NOT EXISTS model TEXT;"))
        conn.execute(text("ALTER TABLE temp_hum_results ADD COLUMN IF NOT EXISTS id_number TEXT;"))


# FULL FILE PROCESSOR

def rebuild_results_for_file(engine, file_path):
    metadata_id, cert_no, doc, meta, action = upsert_metadata_from_docx(engine, file_path)

    identity = {
        "serial_number": meta.get("serial_number"),
        "model": meta.get("model"),
        "id_number": meta.get("id_number"),
    }

    recs = parse_results_from_docx(doc, identity)

    with engine.begin() as conn:
        conn.execute(text("DELETE FROM temp_hum_results WHERE metadata_id = :mid"), {"mid": metadata_id})

    if not recs:
        return {
            "status": "no_results",
            "action": action,
            "metadata_id": metadata_id,
            "certificate_number": cert_no,
            "rows": 0,
            "serial_number": identity.get("serial_number"),
            "model": identity.get("model"),
            "id_number": identity.get("id_number"),
        }

    df = pd.DataFrame(recs)
    df["metadata_id"] = metadata_id
    df["certificate_number"] = cert_no

    df = df[
        [
            "metadata_id",
            "certificate_number",
            "meas_type",
            "chamber_temperature_c",
            "chamber_rh",
            "reference_reading",
            "uut_reading",
            "correction",
            "uncertainty",
            "unit",
            "serial_number",
            "model",
            "id_number",
        ]
    ]

    df.to_sql("temp_hum_results", engine, if_exists="append", index=False)

    return {
        "status": "ok",
        "action": action,
        "metadata_id": metadata_id,
        "certificate_number": cert_no,
        "rows": len(df),
        "serial_number": identity.get("serial_number"),
        "model": identity.get("model"),
        "id_number": identity.get("id_number"),
    }


# MAIN RUNNER

def run_temp_hum_full_import(folder_path=".", ensure_schema=True, max_files=None):
    print("🚀 NML Temp & Hum FULL Import Tool (.docx)")
    print("=" * 78)

    engine = create_engine(DB_URL)

    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("✅ Database connected\n")

    if ensure_schema:
        ensure_metadata_columns(engine)
        ensure_results_schema(engine)

    files = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith(".docx")
        and is_valid_cert_file(f)
    ]

    files = sorted(files)

    if max_files is not None:
        files = files[:max_files]

    if not files:
        print(f"⚠️ No .docx files found in '{folder_path}'")
        return

    print(f"📁 Found {len(files)} .docx files\n")

    results = []
    failures = []

    for i, f in enumerate(files, 1):
        if i == 1 or i % 25 == 0 or i == len(files):
            print(f"⏳ Processing file {i}/{len(files)}: {f}")

        try:
            result = rebuild_results_for_file(engine, os.path.join(folder_path, f))
            result["file_name"] = f
            results.append(result)
        except Exception as e:
            failures.append((f, str(e)))
            print(f"❌ Failed on {f}: {e}")

    created_cnt = sum(1 for r in results if r["action"] == "created")
    updated_cnt = sum(1 for r in results if r["action"] == "updated")
    ok_cnt = sum(1 for r in results if r["status"] == "ok")
    no_results_cnt = sum(1 for r in results if r["status"] == "no_results")
    serial_found_cnt = sum(1 for r in results if r.get("serial_number"))
    model_found_cnt = sum(1 for r in results if r.get("model"))
    id_found_cnt = sum(1 for r in results if r.get("id_number"))

    with engine.connect() as conn:
        meta_cnt = conn.execute(text("SELECT COUNT(*) FROM calibration_metadata")).scalar()
        res_cnt = conn.execute(text("SELECT COUNT(*) FROM temp_hum_results")).scalar()

    print("\n" + "=" * 78)
    print("📊 SUMMARY")
    print("=" * 78)
    print(f"Files found:                     {len(files)}")
    print(f"Processed successfully:         {len(results)}")
    print(f"Metadata rows created:          {created_cnt}")
    print(f"Metadata rows updated:          {updated_cnt}")
    print(f"Certificates with results:      {ok_cnt}")
    print(f"Certificates with no results:   {no_results_cnt}")
    print(f"Serial numbers captured:        {serial_found_cnt}")
    print(f"Models captured:                {model_found_cnt}")
    print(f"ID numbers captured:            {id_found_cnt}")
    print(f"Failures:                       {len(failures)}")
    print(f"public.calibration_metadata:    {meta_cnt} records")
    print(f"public.temp_hum_results:        {res_cnt} records")

    if failures:
        print("\n❌ FAILED FILES")
        print("-" * 78)
        for f, err in failures[:50]:
            print(f"{f}: {err}")

    print("\n✅ Full import complete!")

if __name__ == "__main__":
    # TEST SMALL BATCH :
    #run_temp_hum_full_import(folder_path=IMPORT_FOLDER, ensure_schema=True, max_files=20)

    # RUN FULL FOLDER:
     run_temp_hum_full_import(folder_path=IMPORT_FOLDER, ensure_schema=True, max_files=None)

🚀 NML Temp & Hum FULL Import Tool (.docx)
✅ Database connected

📁 Found 313 .docx files

⏳ Processing file 1/313: 25-0492.docx
⏳ Processing file 25/313: 25-15758.docx
⏳ Processing file 50/313: 25-15952-1.docx
⏳ Processing file 75/313: 26-0032.docx
⏳ Processing file 100/313: 26-0146B.docx
⏳ Processing file 125/313: 26-0251.docx
⏳ Processing file 150/313: 26-0419.docx
⏳ Processing file 175/313: 26-0625.docx
⏳ Processing file 200/313: 26-0720.docx
⏳ Processing file 225/313: 26-0811.docx
⏳ Processing file 250/313: 26-0891.docx
⏳ Processing file 275/313: 26-1002.docx
⏳ Processing file 300/313: 26-1154.docx
⏳ Processing file 313/313: 26-1273.docx

📊 SUMMARY
Files found:                     313
Processed successfully:         313
Metadata rows created:          308
Metadata rows updated:          5
Certificates with results:      291
Certificates with no results:   22
Serial numbers captured:        310
Models captured:                310
ID numbers captured:            310
Failures:         